# Circuit-QED quantum heat engine: photon-assisted Cooper-pair tunnelling

Companion notebook for **Sec. 5** of the QuantumFCS.jl manuscript.

A voltage-biased Josephson junction couples a hot and a cold microwave mode.
Each Cooper pair tunnelling across the junction converts its electrostatic
energy $2eV$ into photons, and tuning the bias to

$$
2eV = l_h \Omega_h - l_c \Omega_c
$$

selects which photon-exchange process is resonant. The manuscript's engine uses
$l_h = 1,\ l_c = 2$: **one hot photon in, two cold photons out**. In the
rotating-wave approximation,

$$
H = -\frac{E_J}{2}\left[\, i^{\,l_h+l_c}\, (a_c^\dagger)^{l_c}\,
    A_h(l_h) A_c(l_c)\, a_h^{l_h} + \text{h.c.} \right],
$$

where the $A_\alpha$ are Laguerre operators carrying the junction's nonlinear
matrix elements. Each mode is damped by its own thermal bath
($\bar n_h, \kappa_h$ and $\bar n_c, \kappa_c$).

We monitor **two** currents — the heat flowing into the hot bath and into the
cold bath — and compute the first two cumulants of each: the mean currents
$\langle J_\alpha\rangle$ and their noise $\langle\langle J_\alpha^2\rangle\rangle$.
From these follow the Fano factors, the entropy production, and the
thermodynamic-uncertainty-relation (TUR) product.

Two regimes are studied:

| | Sec. | regime | question |
|---|---|---|---|
| **Fig. 3** | 5.4 | large affinity | can the engine be *antibunched* ($\mathcal{F}_h<1$) without violating the TUR? |
| **Fig. 4** | 5.5 | finite affinity | how close to the TUR bound $Q = 2$ can it get? |

Both regimes recompute live here. The antibunching sweeps take under a minute;
the finite-affinity sweeps use much larger cutoffs and are behind a toggle.

In [ ]:
using DrWatson
# Locate this repository from the notebook's own position in the tree and
# activate its environment, independently of whichever project the editor
# happens to have selected.
@quickactivate "QuantumFCSNotebooks"

using QuantumToolbox
using QuantumFCS
using Krylov, IncompleteLU
using SparseArrays, LinearAlgebra
using DataFrames, Printf
using CairoMakie, LaTeXStrings

include(srcdir("qhe_model.jl"))
include(srcdir("data_io.jl"))
include(srcdir("paper_figures.jl"))

CairoMakie.activate!(type = "png")
println("project : ", projectdir())
println("Julia ", VERSION)

## Model

`NonlinearQHEParameters` collects the physical parameters and `liouv_RWA` builds
the RWA Hamiltonian together with the four thermal jump operators

$$
J = \Big[\sqrt{(\bar n_h+1)\kappa_h}\,a_h,\ \ \sqrt{(\bar n_c+1)\kappa_c}\,a_c,\ \
        \sqrt{\bar n_h \kappa_h}\,a_h^\dagger,\ \ \sqrt{\bar n_c \kappa_c}\,a_c^\dagger\Big],
$$

i.e. emission into and absorption from each bath. The Josephson energy follows
from the coupling via $E_J = 2g/\big[(2\lambda_h)^{l_h}(2\lambda_c)^{l_c}\big]$.

We start in the **antibunching** regime of Sec. 5.4.

In [ ]:
antibunching = nonlinear_qhe_parameters(
    Nmax_h = 7, Nmax_c = 7,
    lambda_h = 0.47, lambda_c = 0.89,
    Omega_c = 1000.0, Omega_ratio = π,
    kappa_h = 2.0, kappa_c = 0.5,
    n_h = 0.5, n_c = 0.01,
    g = 12.841683366733466,
)

H, J, ws, Ej = liouv_RWA(antibunching)
Ωh = antibunching.Ωratio * antibunching.Ωc

@printf("Hilbert space: %d x %d = %d   (Liouvillian %d x %d)\n",
        antibunching.Nmax_h + 1, antibunching.Nmax_c + 1, prod(ws.dims),
        prod(ws.dims)^2, prod(ws.dims)^2)
@printf("Ω_h = %.2f, Ω_c = %.2f, resonance 2eV = Ω_h - 2Ω_c = %.2f\n",
        Ωh, antibunching.Ωc, Ωh - 2*antibunching.Ωc)
@printf("g = %.4f  ->  E_J = %.4f\n", antibunching.g, Ej)
@printf("%d jump operators (hot/cold emission and absorption)\n", length(J))

## Two currents, one factorization

This is the part of the API worth seeing. Both heat currents live on the *same*
Liouvillian and the *same* steady state — only the monitored jumps and their
weights differ:

* **hot current**: jumps `[J₁, J₃]` (emission into / absorption from the hot
  bath) with weights `[-Ω_h, +Ω_h]`;
* **cold current**: jumps `[J₂, J₄]` with weights `[-Ω_c, +Ω_c]`.

So we prepare the Drazin solver **once** with `prepare_fcs_context`, then call
`fcscumulants_recursive` on that context for each current. Passing the
`TraceConstrainedSteadyState` straight into `prepare_fcs_context` also hands
over the preconditioner built during the steady-state solve, so the whole point
costs a single factorization no matter how many currents we ask for.

In [ ]:
numerics = nonlinear_qhe_numerics(solver = :iterative, trunc_tol = 5e-3,
                                  occupation_tol = 0.5)

# Steady state (returns the ILU preconditioner alongside rho_ss).
steady = qhe_steadystate(H, J, ws, antibunching, numerics)

# Prepare the Drazin solver once, reusing the steady-state preconditioner.
ctx = prepare_fcs_context(steady.fcs_steadystate;
                          method = :iterative,
                          τ      = numerics.fcs_tau,
                          rtol   = numerics.fcs_rtol,
                          itmax  = numerics.fcs_itmax,
                          memory = numerics.fcs_memory)

# Same context, two different monitored currents.
hot  = fcscumulants_recursive(ctx; mJ = [J[1], J[3]], nu = [-Ωh, Ωh], nC = 2)
cold = fcscumulants_recursive(ctx; mJ = [J[2], J[4]],
                              nu = [-antibunching.Ωc, antibunching.Ωc], nC = 2)

Jh, Dh = real(hot[1]),  real(hot[2])
Jc, Dc = real(cold[1]), real(cold[2])

@printf("hot :  ⟨J_h⟩ = %12.6g   ⟨⟨J_h²⟩⟩ = %12.6g\n", Jh, Dh)
@printf("cold:  ⟨J_c⟩ = %12.6g   ⟨⟨J_c²⟩⟩ = %12.6g\n", Jc, Dc)
@printf("steady state converged in %d iterations\n", steady.ss_iterations)

### Consistency checks

Two identities must hold, and both are checked at every point of the sweeps.

**Tight coupling.** Because every tunnelling event moves exactly $l_h$ hot and
$l_c$ cold photons, the two heat currents are rigidly locked:

$$
l_c \Omega_c \langle J_h\rangle + l_h \Omega_h \langle J_c\rangle = 0 .
$$

**Current identity.** The first FCS cumulant must equal the current computed
directly from the steady state as a weighted sum of jump rates,
$\sum_k \nu_k \langle J_k^\dagger J_k\rangle$ — an independent route to the same
number.

In [ ]:
tight_coupling_error = abs(antibunching.lc * antibunching.Ωc * Jh +
                           antibunching.lh * Ωh * Jc) /
    max(abs(antibunching.lc * antibunching.Ωc * Jh),
        abs(antibunching.lh * Ωh * Jc), eps())

hot_direct  = qhe_jump_current(steady.ρss, [J[1], J[3]], [-Ωh, Ωh])
cold_direct = qhe_jump_current(steady.ρss, [J[2], J[4]],
                               [-antibunching.Ωc, antibunching.Ωc])

@printf("tight coupling  l_c Ω_c ⟨J_h⟩ + l_h Ω_h ⟨J_c⟩ = 0  ->  rel. error %.2e\n",
        tight_coupling_error)
@printf("hot  current identity  c₁/direct − 1 = %.2e\n", Jh/hot_direct - 1)
@printf("cold current identity  c₁/direct − 1 = %.2e\n", Jc/cold_direct - 1)

@assert tight_coupling_error < 1e-5
@assert isapprox(Jh, hot_direct;  rtol = 1e-4)
@assert isapprox(Jc, cold_direct; rtol = 1e-4)

## Thermodynamics: affinity, Fano factors, and the TUR

The thermodynamic force driving the engine is the affinity

$$
\mathcal{A} = \frac{1}{l_h}\ln\!\left(\frac{1}{\bar n_c}+1\right)
            - \frac{1}{l_c}\ln\!\left(\frac{1}{\bar n_h}+1\right),
$$

which sets the entropy production $\sigma_\alpha$. The TUR states that the
uncertainty product obeys

$$
Q_\alpha = \frac{\langle\langle J_\alpha^2\rangle\rangle}{\langle J_\alpha\rangle^2}\,\sigma_\alpha \;\ge\; 2 .
$$

Rewriting it as a bound on the Fano factor
$\mathcal{F}_\alpha = \langle\langle J_\alpha^2\rangle\rangle/(\Omega_\alpha|\langle J_\alpha\rangle|)$
gives a threshold $\mathcal{F}^{\mathrm{TUR}}$. The key point of Sec. 5.4: at
large affinity that threshold drops **well below** the Poisson value 1, so an
engine can be strongly sub-Poissonian — antibunched — and still respect the TUR.

`qhe_point` assembles all of this for one parameter set.

In [ ]:
pt = qhe_point(antibunching, numerics)

@printf("affinity        𝒜 = %.4f\n", pt.A)
@printf("hot  Fano       ℱ_h = %.4f      TUR threshold ℱ_h^TUR = %.4f\n",
        pt.Fh, pt.Fh_TUR_threshold)
@printf("cold Fano       ℱ_c = %.4f      TUR threshold ℱ_c^TUR = %.4f\n",
        pt.Fc, pt.Fc_TUR_threshold)
@printf("uncertainty     Q_h = %.4f,  Q_c = %.4f      (TUR bound: 2)\n", pt.Qh, pt.Qc)
@printf("output power    ⟨P⟩ = %.6g\n", pt.P_drive)
@printf("RWA coherence   𝒞_RWA = %.4f,  off-resonant ε_off = %.2e\n", pt.CRWA, pt.epsilon_off)

if pt.Fh < 1 && pt.Qh > 2
    println("\n→ sub-Poissonian (ℱ_h < 1) yet Q_h > 2: antibunching without TUR violation.")
end

# The prepared-context route above and qhe_point agree.
@printf("\nprepared-context vs qhe_point:  |ΔJ_h|/J_h = %.2e,  |ΔD_h|/D_h = %.2e\n",
        abs(Jh/pt.Jh - 1), abs(Dh/pt.Dh - 1))

## Antibunching regime (Fig. 3)

Two sweeps at fixed bath parameters: one over the coupling $g$ at
$\lambda_c = 0.89$, one over $\lambda_c$ at the selected $g$. The $\lambda_c$
sweep needs a slightly larger cold cutoff.

Both run live — 500 and 100 points, well under a minute in total. Progress is
printed periodically rather than per point, since the points are fast.

In [ ]:
# Wrapped in a function: keeps the loop type-stable and lets a sweep be rerun.
function qhe_sweep(values, setter; progress_every = 50, label = "sweep", kwargs...)
    rows = NamedTuple[]
    t0 = time_ns()
    for (i, v) in enumerate(values)
        pt = qhe_point(; kwargs..., setter(v)...)
        push!(rows, pt)
        if i == 1 || i == length(values) || i % progress_every == 0
            @printf("%-14s %3d/%3d  x=%8.4f | ℱ_h=%7.4f ℱ_c=%7.4f Q_h=%7.3f | tails %.1e/%.1e | %5.1fs\n",
                    label, i, length(values), v, pt.Fh, pt.Fc, pt.Qh,
                    pt.hot_tail, pt.cold_tail, (time_ns()-t0)/1e9)
            flush(stdout)
        end
    end
    return DataFrame(rows)
end

antibunching_common = (; lambda_h = 0.47, Omega_c = 1000.0, Omega_ratio = π,
                         kappa_h = 2.0, kappa_c = 0.5, n_h = 0.5, n_c = 0.01,
                         solver = :iterative, trunc_tol = 5e-3, occupation_tol = 0.5)

df_antibunching_g = qhe_sweep(LinRange(1.0, 20.0, 500), v -> (; g = v);
    label = "antibunch g", progress_every = 100,
    Nmax_h = 7, Nmax_c = 7, lambda_c = 0.89, antibunching_common...)

df_antibunching_λc = qhe_sweep(LinRange(0.1, 2.0, 100), v -> (; λc = v);
    label = "antibunch λc", progress_every = 25,
    Nmax_h = 7, Nmax_c = 10, g = 12.841683366733466, antibunching_common...)

@printf("\nmin ℱ_h over the g sweep: %.4f at g = %.3f  (TUR threshold %.4f)\n",
        minimum(df_antibunching_g.Fh),
        df_antibunching_g.g[argmin(df_antibunching_g.Fh)],
        first(df_antibunching_g.Fh_TUR_threshold))

The hot Fano factor dips to $\approx 0.59$ — clearly sub-Poissonian — while the
TUR threshold sits near $0.25$. The engine is antibunched with room to spare
above the bound.

`qhe_antibunching_paper_figure` is the exact routine behind Fig. 3: scaled heat
currents and output power (a, e), hot and cold Fano factors against their TUR
thresholds and the Poisson line (b, c, f, g), and the RWA coherence (d, h).

In [ ]:
fig3 = qhe_antibunching_paper_figure(df_antibunching_g, df_antibunching_λc)
save(projectdir("figures", "qhe_antibunching.png"), fig3; px_per_unit = 2)
fig3

### Validation

The RWA and the Fock truncation both have to be checked before the physics can
be believed. The validation figure reports, across both sweeps:

* $\varepsilon_{\text{off}}$ — the size of the neglected off-resonant terms
  relative to their detuning; the RWA needs this $\ll 1$;
* the boundary populations of the hot and cold marginals against the truncation
  tolerance;
* the occupation fractions against the cutoff cap.

The summary below is deliberately blunt about where the model stops being
trustworthy, rather than reporting a single pass/fail.

In [ ]:
fig3v = qhe_antibunching_validation_figure(df_antibunching_g, df_antibunching_λc)
save(projectdir("figures", "qhe_antibunching_validation.png"), fig3v; px_per_unit = 2)

function validation_summary(df, xs, label; epsilon_tol = 0.05)
    ok = df.epsilon_off .< epsilon_tol
    @printf("%-9s | %3d/%3d points satisfy ε_off < %.2f | ε_off ∈ [%.1e, %.1e]\n",
            label, count(ok), nrow(df), epsilon_tol,
            minimum(df.epsilon_off), maximum(df.epsilon_off))
    @printf("          | tails: hot ≤ %.1e, cold ≤ %.1e | tight coupling ≤ %.1e | cutoffs ok: %s\n",
            maximum(df.hot_tail), maximum(df.cold_tail),
            maximum(df.tight_coupling_error), all(df.cutoff_ok))
    if !all(ok)
        @printf("          | RWA marginal for x ∈ [%.3f, %.3f]\n",
                minimum(xs[.!ok]), maximum(xs[.!ok]))
    end
end

validation_summary(df_antibunching_g,  df_antibunching_g.g,   "g sweep")
validation_summary(df_antibunching_λc, df_antibunching_λc.λc, "λc sweep")
fig3v

The $g$ sweep is clean throughout: $\varepsilon_{\text{off}}$ stays below
$10^{-2}$ at every point, truncation tails are at the $10^{-3}$ level set by
`trunc_tol`, and tight coupling holds to $\sim\!10^{-6}$.

The $\lambda_c$ sweep is only trustworthy above $\lambda_c \approx 0.37$. This
is not a numerical problem but a physical one: holding $g$ fixed while shrinking
$\lambda_c$ requires $E_J = g/(4\lambda_h\lambda_c^2)$ to grow without bound, so
the off-resonant terms the RWA discards stop being small. The cutoffs remain
perfectly adequate there — it is the *model*, not the solver, that degrades.
The operating point used for the manuscript's claims, $\lambda_c = 0.89$, sits
comfortably inside the valid region.

## Finite-affinity regime (Fig. 4)

Lowering the affinity (hotter cold bath: $\bar n_h = 2.5$, $\bar n_c = 1.5$)
raises the TUR thresholds far above 1, and the engine can then be pushed close
to the bound $Q \approx 2$ — near-reversible operation with small currents.

This regime needs much larger cutoffs ($N_{\max,h} = 20$, $N_{\max,c} = 25$, a
546-dimensional Hilbert space), so each point costs a few seconds and the two
sweeps together take roughly **eight minutes**. They are therefore behind a
toggle; by default the checked-in sweeps are loaded. Set
`RUN_FINITE_AFFINITY_LIVE = true` to recompute them — the results are identical.

In [ ]:
RUN_FINITE_AFFINITY_LIVE = false     # ← true recomputes both sweeps (~8 minutes)

finite_affinity_common = (; lambda_h = 0.47, Omega_c = 1000.0, Omega_ratio = π,
                            kappa_h = 2.0, kappa_c = 0.5, n_h = 2.5, n_c = 1.5,
                            solver = :iterative, trunc_tol = 3e-3, occupation_tol = 0.5)

if RUN_FINITE_AFFINITY_LIVE
    df_finite_affinity_g = qhe_sweep(LinRange(0.1, 100.0, 50), v -> (; g = v);
        label = "affinity g", progress_every = 5,
        Nmax_h = 20, Nmax_c = 25, lambda_c = 0.7, finite_affinity_common...)
    df_finite_affinity_λc = qhe_sweep(LinRange(0.1, 2.0, 50), v -> (; λc = v);
        label = "affinity λc", progress_every = 5,
        Nmax_h = 20, Nmax_c = 25, g = 10.0, finite_affinity_common...)
else
    df_finite_affinity_g,  _ = load_qhe_sweep("qhe_finite_affinity_g")
    df_finite_affinity_λc, _ = load_qhe_sweep("qhe_finite_affinity_lambda_c")
    println("loaded checked-in finite-affinity sweeps")
end

best = argmin(abs.(df_finite_affinity_g.Qh .- 2))
@printf("affinity 𝒜 = %.4f   (antibunching regime had 𝒜 = %.4f)\n",
        first(df_finite_affinity_g.A), pt.A)
@printf("TUR thresholds: ℱ_h^TUR = %.3f, ℱ_c^TUR = %.3f\n",
        first(df_finite_affinity_g.Fh_TUR_threshold),
        first(df_finite_affinity_g.Fc_TUR_threshold))
@printf("closest approach to the bound: Q_h = %.4f at g = %.3f\n",
        df_finite_affinity_g.Qh[best], df_finite_affinity_g.g[best])

In [ ]:
fig4 = qhe_finite_affinity_paper_figure(df_finite_affinity_g, df_finite_affinity_λc)
save(projectdir("figures", "qhe_finite_affinity_tur.png"), fig4; px_per_unit = 2)
fig4

In [ ]:
fig4v = qhe_finite_affinity_validation_figure(df_finite_affinity_g, df_finite_affinity_λc)
save(projectdir("figures", "qhe_finite_affinity_validation.png"), fig4v; px_per_unit = 2)
fig4v

## Cutoff convergence

The Fock cutoffs above are fixed rather than scheduled, so they need justifying.
`qhe_cumulant_convergence_table` recomputes a point on a ladder of cutoffs and
reports the relative change in the currents and Fano factors from one rung to
the next. Once those changes fall well below the precision claimed for the
figures, the cutoff is adequate.

In [ ]:
convergence = qhe_cumulant_convergence_table(
    [(5, 5), (7, 7), (9, 9), (11, 11)];
    lambda_h = 0.47, lambda_c = 0.89, Omega_c = 1000.0, Omega_ratio = π,
    kappa_h = 2.0, kappa_c = 0.5, n_h = 0.5, n_c = 0.01,
    g = 12.841683366733466, solver = :iterative,
    trunc_tol = 5e-3, occupation_tol = 0.5)

convergence[:, [:Nmax_h, :Nmax_c, :Jh, :Fh, :Fc,
                :hot_current_rel_change, :hot_Fh_rel_change, :cold_Fc_rel_change,
                :hot_tail, :cold_tail]]

The changes fall by orders of magnitude as the cutoff grows, and the production
cutoff $N_{\max} = 7$ used for Fig. 3 already sits in the converged region.

---

**Summary.** At large affinity the engine is antibunched
($\mathcal{F}_h \approx 0.59 < 1$) while remaining comfortably above its TUR
threshold ($\approx 0.25$) — sub-Poissonian noise is not by itself a TUR
violation. At finite affinity the thresholds rise above 1 and the engine
approaches $Q \approx 2$, saturating the classical bound.